In [1]:
# Cell: Import & cấu hình chung
import os, json, glob, shutil, random, math, time
from pathlib import Path
from typing import List, Dict, Any, Tuple

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.ops import nms
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2
from ultralytics import YOLO
from tqdm import tqdm

SEED = 2025
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True

# Đường dẫn gốc
ROOT = Path.cwd()

# ĐÚNG VỚI DỮ LIỆU CỦA BẠN
DATA_OBS = ROOT / "observing" / "train"               # thư mục train
DATA_TEST = ROOT / "public_test" / "public_test"      # thư mục test
WORKDIR = ROOT / "workdir_one_shot"                   # nơi sinh dữ liệu trung gian
WORKDIR.mkdir(parents=True, exist_ok=True)

ANN_PATH = DATA_OBS / "annotations" / "annotations.json"
SAMPLES_DIR = DATA_OBS / "samples"
TEST_SAMPLES_DIR = DATA_TEST / "samples"

assert ANN_PATH.exists(), f"Không tìm thấy {ANN_PATH}"
assert SAMPLES_DIR.exists(), f"Không tìm thấy {SAMPLES_DIR}"
assert TEST_SAMPLES_DIR.exists(), f"Không tìm thấy {TEST_SAMPLES_DIR}"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

c:\Users\grazt\anaconda3\envs\AI\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


In [2]:
# Cell: Đọc annotations + in thống kê đúng schema
with open(ANN_PATH, "r", encoding="utf-8") as f:
    annotations = json.load(f)

def count_unique_frames(entry: Dict[str, Any]) -> int:
    frames = set()
    for ann_set in entry.get("annotations", []):
        for bb in ann_set.get("bboxes", []):
            frames.add(int(bb["frame"]))
    return len(frames)

for item in annotations:
    vid = item.get("video_id")
    nf = count_unique_frames(item)
    print(f"video_id: {vid}\n\nnum_frames: {nf}\n\n" + "-"*60 + "\n")

video_id: Backpack_0

num_frames: 3184

------------------------------------------------------------

video_id: Backpack_1

num_frames: 1454

------------------------------------------------------------

video_id: Jacket_0

num_frames: 1162

------------------------------------------------------------

video_id: Jacket_1

num_frames: 690

------------------------------------------------------------

video_id: Laptop_0

num_frames: 884

------------------------------------------------------------

video_id: Laptop_1

num_frames: 987

------------------------------------------------------------

video_id: Lifering_0

num_frames: 1134

------------------------------------------------------------

video_id: Lifering_1

num_frames: 1511

------------------------------------------------------------

video_id: MobilePhone_0

num_frames: 968

------------------------------------------------------------

video_id: MobilePhone_1

num_frames: 889

-------------------------------------------------

In [3]:
# Cell: Liệt kê video_ids có trong samples train
all_video_ids = [p.name for p in sorted(SAMPLES_DIR.glob("*")) if (p / "drone_video.mp4").exists()]
print("Tổng video train+val (samples):", len(all_video_ids))

Tổng video train+val (samples): 14


In [4]:
# Cell: Chia train/val theo video (80/20)
random.shuffle(all_video_ids)
split_ratio = 0.8
n_train = int(len(all_video_ids) * split_ratio)
train_ids = set(all_video_ids[:n_train])
val_ids = set(all_video_ids[n_train:])

print(f"Tổng video: {len(all_video_ids)} | Train: {len(train_ids)} | Val: {len(val_ids)}")

Tổng video: 14 | Train: 11 | Val: 3


In [5]:
# Cell: Adapter từ schema thật -> frame -> list(x,y,w,h)
def get_video_frames_and_boxes(video_id: str) -> Dict[int, List[List[float]]]:
    # Tìm entry theo video_id trong annotations
    entry = next((x for x in annotations if x.get("video_id") == video_id), None)
    assert entry is not None, f"video_id {video_id} không có trong annotations"
    mapping = {}
    for ann_set in entry.get("annotations", []):
        for bb in ann_set.get("bboxes", []):
            fidx = int(bb["frame"])
            x1, y1, x2, y2 = int(bb["x1"]), int(bb["y1"]), int(bb["x2"]), int(bb["y2"])
            w = max(1, x2 - x1)
            h = max(1, y2 - y1)
            mapping.setdefault(fidx, []).append([x1, y1, w, h])
    return mapping

def yolo_bbox_from_xywh(x, y, w, h, img_w, img_h):
    cx = (x + w/2) / img_w
    cy = (y + h/2) / img_h
    bw = w / img_w
    bh = h / img_h
    return cx, cy, bw, bh

In [6]:
# Cell: Chuẩn bị thư mục YOLO
IMG_TRAIN = WORKDIR / "yolo_dataset" / "images" / "train"
IMG_VAL   = WORKDIR / "yolo_dataset" / "images" / "val"
LBL_TRAIN = WORKDIR / "yolo_dataset" / "labels" / "train"
LBL_VAL   = WORKDIR / "yolo_dataset" / "labels" / "val"
for p in [IMG_TRAIN, IMG_VAL, LBL_TRAIN, LBL_VAL]:
    p.mkdir(parents=True, exist_ok=True)

In [10]:
# Cell: Trích xuất frames + ghi nhãn YOLO từ video train/val
def extract_frames_and_labels(video_id: str, split: str):
    video_path = SAMPLES_DIR / video_id / "drone_video.mp4"
    assert video_path.exists(), f"Thiếu video: {video_path}"
    cap = cv2.VideoCapture(str(video_path))
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    img_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    img_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_boxes = get_video_frames_and_boxes(video_id)

    img_dir = IMG_TRAIN if split=="train" else IMG_VAL
    lbl_dir = LBL_TRAIN if split=="train" else LBL_VAL

    idx = 0
    success, frame = cap.read()
    while success:
        out_name = f"{video_id}_f{idx:06d}"
        img_out = img_dir / f"{out_name}.jpg"
        lbl_out = lbl_dir / f"{out_name}.txt"

        # Ghi ảnh
        cv2.imwrite(str(img_out), frame)

        # Ghi label
        boxes = frame_boxes.get(idx, [])
        lines = []
        for b in boxes:
            x,y,w,h = b  # pixels
            cx, cy, bw, bh = yolo_bbox_from_xywh(x,y,w,h,img_w,img_h)
            lines.append(f"0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
        with open(lbl_out, "w") as f:
            f.write("\n".join(lines))

        idx += 1
        success, frame = cap.read()

    cap.release()
    return n_frames

# Thực hiện trích xuất
for vid in tqdm(all_video_ids, desc="Extracting"):
    split = "train" if vid in train_ids else "val"
    extract_frames_and_labels(vid, split)

print("Số ảnh train:", len(list(IMG_TRAIN.glob("*.jpg"))),
      "| Số nhãn train:", len(list(LBL_TRAIN.glob("*.txt"))))
print("Số ảnh val:", len(list(IMG_VAL.glob("*.jpg"))),
      "| Số nhãn val:", len(list(LBL_VAL.glob("*.txt"))))

Extracting: 100%|██████████| 14/14 [05:46<00:00, 24.77s/it]


Số ảnh train: 64210 | Số nhãn train: 64210
Số ảnh val: 16519 | Số nhãn val: 16519


In [11]:
# Cell: Tạo YAML dataset cho YOLOv8
DATA_YAML = WORKDIR / "yolo_dataset" / "one_shot_dataset.yaml"
DATA_YAML.parent.mkdir(parents=True, exist_ok=True)

yaml_text = f"""path: {str((WORKDIR / "yolo_dataset").as_posix())}
train: images/train
val: images/val
nc: 1
names: ["target"]
"""
with open(DATA_YAML, "w") as f:
    f.write(yaml_text)

print("YAML dataset path:", DATA_YAML)

YAML dataset path: d:\zalo.ai\AeroEyes\workdir_one_shot\yolo_dataset\one_shot_dataset.yaml


In [12]:
# Cell: Huấn luyện YOLOv8s
yolo_model_name = "yolov8s.pt"  
exp_dir = WORKDIR / "yolo_train"
exp_dir.mkdir(exist_ok=True, parents=True)

yolo = YOLO(yolo_model_name)
results = yolo.train(
    data=str(DATA_YAML),
    project=str(exp_dir),
    name="exp_one_shot",
    epochs=5,          
    imgsz=640,
    batch=16,
    lr0=0.002,
    optimizer="SGD",
    weight_decay=5e-4,
    mosaic=1.0,           # Mosaic
    mixup=0.2,            # MixUp
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    fliplr=0.5, flipud=0.0,
    degrees=10.0, perspective=0.001, shear=0.0, translate=0.1, scale=0.9,
    patience=20,
)
best_yolo_ckpt = Path(results.save_dir) / "weights" / "best.pt"
print("Best YOLO weights:", best_yolo_ckpt)

New https://pypi.org/project/ultralytics/8.3.227 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.221  Python-3.10.18 torch-2.9.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5070 Ti, 16302MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=d:\zalo.ai\AeroEyes\workdir_one_shot\yolo_dataset\one_shot_dataset.yaml, degrees=10.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.002, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, 

In [72]:
# Cell: Augmentations cho embedding
IMG_SIZE = 224

embed_train_aug = A.Compose([
    # --- Tăng cường Augmentation Hình học (Geometric) ---
    A.LongestMaxSize(max_size=IMG_SIZE),
    A.PadIfNeeded(min_height=IMG_SIZE, min_width=IMG_SIZE, border_mode=cv2.BORDER_CONSTANT, value=114),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=25, border_mode=cv2.BORDER_REFLECT_101, p=0.8), # Tăng rotate và p
    A.Perspective(scale=(0.05, 0.1), pad_mode=cv2.BORDER_REFLECT_101, p=0.2), # Thêm Perspective
    A.HorizontalFlip(p=0.5),
    
    # --- Giữ nguyên Augmentation Màu sắc đã sửa ---
    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.3), 
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.2),
    ToTensorV2()
])

embed_infer_aug = A.Compose([
    A.LongestMaxSize(max_size=IMG_SIZE),
    A.PadIfNeeded(min_height=IMG_SIZE, min_width=IMG_SIZE, border_mode=cv2.BORDER_CONSTANT, value=114),
    ToTensorV2()
])

C:\Users\grazt\AppData\Local\Temp\ipykernel_15120\2968886710.py:7: UserWarning: Argument(s) 'value' are not valid for transform PadIfNeeded
  A.PadIfNeeded(min_height=IMG_SIZE, min_width=IMG_SIZE, border_mode=cv2.BORDER_CONSTANT, value=114),
C:\Users\grazt\AppData\Local\Temp\ipykernel_15120\2968886710.py:9: UserWarning: Argument(s) 'pad_mode' are not valid for transform Perspective
  A.Perspective(scale=(0.05, 0.1), pad_mode=cv2.BORDER_REFLECT_101, p=0.2), # Thêm Perspective
C:\Users\grazt\AppData\Local\Temp\ipykernel_15120\2968886710.py:14: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=0.2),
C:\Users\grazt\AppData\Local\Temp\ipykernel_15120\2968886710.py:20: UserWarning: Argument(s) 'value' are not valid for transform PadIfNeeded
  A.PadIfNeeded(min_height=IMG_SIZE, min_width=IMG_SIZE, border_mode=cv2.BORDER_CONSTANT, value=114),


In [61]:
# Cell: Tiện ích nạp ảnh tham chiếu train
def load_ref_images_train(video_id: str) -> List[np.ndarray]:
    odir = SAMPLES_DIR / video_id / "object_images"
    imgs = []
    for k in [1,2,3]:
        p = odir / f"img_{k}.jpg"
        if p.exists():
            imgs.append(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB))
    return imgs

def crop_xywh(img: np.ndarray, box: List[float]) -> np.ndarray:
    x,y,w,h = map(int, box)
    H, W = img.shape[:2]
    x2 = min(W, x+w); y2 = min(H, y+h)
    x1 = max(0, x); y1 = max(0, y)
    return img[y1:y2, x1:x2, :]

In [73]:
# Cell: Dataset sinh (ref, pos, neg)
class SiamesePairs(Dataset):
    def __init__(self, split_ids: List[str], all_video_ids: List[str], max_neg_per_video=3, samples_per_video=20):
        self.items = []
        
        # 1. Tạo một "kho" chứa tất cả các vật thể (positives) từ TẤT CẢ video
        #    để dùng làm "hard negatives"
        self.positive_pool = {} # Key: video_id, Value: List[Tuple(frame_idx, box)]
        all_vids_for_pool = list(all_video_ids) # Sử dụng tất cả video
        print("Building positive pool for hard negative mining...")
        for vid in tqdm(all_vids_for_pool, desc="Pooling positives"):
            frame_map = get_video_frames_and_boxes(vid)
            if not frame_map:
                continue
            self.positive_pool[vid] = []
            for fidx, boxes in frame_map.items():
                for box in boxes[:1]: # Chỉ lấy 1 box mỗi frame
                    self.positive_pool[vid].append((fidx, box))

        # 2. Tạo triplets
        print("Generating Siamese triplets...")
        for vid in tqdm(split_ids, desc="Generating triplets"):
            if vid not in self.positive_pool or not self.positive_pool[vid]:
                continue
                
            refs = load_ref_images_train(vid)
            if not refs: continue
            
            vpath = SAMPLES_DIR / vid / "drone_video.mp4"
            cap = cv2.VideoCapture(str(vpath))
            if not cap.isOpened(): continue
            
            # Lấy "category" của video hiện tại (ví dụ: "Backpack")
            current_category = vid.split('_')[0]
            
            # Tìm tất cả video_id KHÁC category
            hard_negative_vids = [
                v for v in all_vids_for_pool 
                if v in self.positive_pool and v.split('_')[0] != current_category
            ]
            if not hard_negative_vids: # Fallback nếu chỉ có 1 category
                hard_negative_vids = [v for v in all_vids_for_pool if v != vid and v in self.positive_pool]
            if not hard_negative_vids:
                print(f"Warning: No hard negative videos found for {vid}. Skipping.")
                cap.release()
                continue

            # Lấy các mẫu "positive" từ video này
            pos_samples = self.positive_pool[vid]
            chosen_pos_samples = random.sample(pos_samples, k=min(samples_per_video, len(pos_samples)))
            
            # Cache cho hard negative video
            current_neg_cap = None
            current_neg_vid = None

            for (fidx, pos_box) in chosen_pos_samples:
                # Lấy ảnh POSITIVE
                cap.set(cv2.CAP_PROP_POS_FRAMES, fidx)
                ok, frame = cap.read()
                if not ok: continue
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                pos_img = crop_xywh(frame_rgb, pos_box)
                
                # Lấy ảnh HARD NEGATIVE
                for _ in range(max_neg_per_video):
                    neg_img = None
                    try:
                        # Chọn 1 video hard negative ngẫu nhiên
                        neg_vid_id = random.choice(hard_negative_vids)
                        
                        # Chọn 1 mẫu ngẫu nhiên từ video đó
                        neg_fidx, neg_box = random.choice(self.positive_pool[neg_vid_id])
                        
                        # Mở video hard negative (hoặc dùng lại nếu đã cache)
                        if current_neg_vid != neg_vid_id:
                            if current_neg_cap: current_neg_cap.release()
                            current_neg_cap = cv2.VideoCapture(str(SAMPLES_DIR / neg_vid_id / "drone_video.mp4"))
                            current_neg_vid = neg_vid_id
                        
                        if not current_neg_cap or not current_neg_cap.isOpened():
                            current_neg_vid = None
                            continue

                        current_neg_cap.set(cv2.CAP_PROP_POS_FRAMES, neg_fidx)
                        ok_neg, neg_frame = current_neg_cap.read()
                        if ok_neg:
                            neg_frame_rgb = cv2.cvtColor(neg_frame, cv2.COLOR_BGR2RGB)
                            neg_img = crop_xywh(neg_frame_rgb, neg_box)
                            self.items.append((random.choice(refs), pos_img, neg_img))
                    except Exception as e:
                        continue # Bỏ qua nếu lỗi
            
            cap.release()
            if current_neg_cap:
                current_neg_cap.release()

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        ref, pos, neg = self.items[idx]
        ref_t = embed_train_aug(image=ref)["image"]
        pos_t = embed_train_aug(image=pos)["image"]
        neg_t = embed_train_aug(image=neg)["image"]
        return ref_t, pos_t, neg_t

train_siamese = SiamesePairs(list(train_ids), all_video_ids, max_neg_per_video=3, samples_per_video=20)
val_siamese = SiamesePairs(list(val_ids), all_video_ids, max_neg_per_video=2, samples_per_video=10)

dl_train_siamese = DataLoader(train_siamese, batch_size=32, shuffle=True, num_workers=0, pin_memory=False, drop_last=True)
dl_val_siamese = DataLoader(val_siamese, batch_size=32, shuffle=False, num_workers=0, pin_memory=False)
print("Siamese train items:", len(train_siamese), "| val items:", len(val_siamese))

Building positive pool for hard negative mining...


Pooling positives: 100%|██████████| 14/14 [00:00<00:00, 86.94it/s]


Generating Siamese triplets...


Generating triplets: 100%|██████████| 11/11 [00:57<00:00,  5.22s/it]


Building positive pool for hard negative mining...


Pooling positives: 100%|██████████| 14/14 [00:00<00:00, 696.40it/s]


Generating Siamese triplets...


Generating triplets: 100%|██████████| 3/3 [00:05<00:00,  1.88s/it]

Siamese train items: 660 | val items: 60


In [74]:
# Cell: Mạng embedding (ResNet18 + projection 256-d)
class SiameseEmbedding(nn.Module):
    def __init__(self, out_dim=256, pretrained=True):
        super().__init__()
        base = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
        feat_dim = base.fc.in_features
        base.fc = nn.Identity()
        self.backbone = base
        self.head = nn.Sequential(
            nn.Linear(feat_dim, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3), # THÊM DROPOUT
            nn.Linear(512, out_dim)
        )

    def forward(self, x):
        f = self.backbone(x)
        z = self.head(f)
        z = F.normalize(z, dim=-1)
        return z

embed_dim = 256
embed_model = SiameseEmbedding(out_dim=embed_dim, pretrained=True).to(DEVICE)

In [ ]:
# Cell: Train embedding với CosineEmbeddingLoss
optimizer = torch.optim.AdamW(embed_model.parameters(), lr=1e-4, weight_decay=1e-5) # Giảm LR
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

# SỬ DỤNG TripletMarginLoss
triplet_loss = nn.TripletMarginLoss(margin=0.3, p=2.0) # p=2.0 là L2 distance (Euclidean)

def run_epoch(dloader, train=True):
    embed_model.train(train)
    total_loss = 0.0
    for ref, pos, neg in tqdm(dloader, leave=False):
        ref = ref.to(DEVICE).float()/255.0
        pos = pos.to(DEVICE).float()/255.0
        neg = neg.to(DEVICE).float()/255.0
        with torch.set_grad_enabled(train):
            zr = embed_model(ref) # Anchor
            zp = embed_model(pos) # Positive
            zn = embed_model(neg) # Negative
            
            # Tính loss
            loss = triplet_loss(zr, zp, zn)
            
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        total_loss += loss.item()
    return total_loss / max(1, len(dloader))

best_val = 1e9
EPOCHS_EMB = 10
best_embed_path = WORKDIR / "siamese_embed_best.pt"

for e in range(EPOCHS_EMB):
    tl = run_epoch(dl_train_siamese, train=True)
    vl = run_epoch(dl_val_siamese, train=False) if len(val_siamese)>0 else tl
    scheduler.step()
    print(f"[Embed] Epoch {e+1}/{EPOCHS_EMB} | train_loss={tl:.4f} | val_loss={vl:.4f}")
    if vl < best_val:
        best_val = vl
        torch.save(embed_model.state_dict(), best_embed_path)
        print("Saved:", best_embed_path)

[Embed] Epoch 1/10 | train_loss=0.1989 | val_loss=0.1746
Saved: d:\zalo.ai\AeroEyes\workdir_one_shot\siamese_embed_best.pt


[Embed] Epoch 2/10 | train_loss=0.0519 | val_loss=0.1205
Saved: d:\zalo.ai\AeroEyes\workdir_one_shot\siamese_embed_best.pt


[Embed] Epoch 3/10 | train_loss=0.0212 | val_loss=0.0643
Saved: d:\zalo.ai\AeroEyes\workdir_one_shot\siamese_embed_best.pt


[Embed] Epoch 4/10 | train_loss=0.0105 | val_loss=0.0591
Saved: d:\zalo.ai\AeroEyes\workdir_one_shot\siamese_embed_best.pt


[Embed] Epoch 5/10 | train_loss=0.0063 | val_loss=0.0413
Saved: d:\zalo.ai\AeroEyes\workdir_one_shot\siamese_embed_best.pt


[Embed] Epoch 6/10 | train_loss=0.0037 | val_loss=0.0368
Saved: d:\zalo.ai\AeroEyes\workdir_one_shot\siamese_embed_best.pt


[Embed] Epoch 7/10 | train_loss=0.0068 | val_loss=0.0354
Saved: d:\zalo.ai\AeroEyes\workdir_one_shot\siamese_embed_best.pt


[Embed] Epoch 8/10 | train_loss=0.0054 | val_loss=0.0438


[Embed] Epoch 9/10 | train_loss=0.0030 | val_loss=0.0347
Saved: d:\zalo.ai\AeroEyes\workdir_one_shot\siamese_embed_best.pt


[Embed] Epoch 10/10 | train_loss=0.0055 | val_loss=0.0476


In [76]:
# Cell: Tiện ích nạp ảnh tham chiếu TEST
def load_ref_images_test(video_id: str) -> List[np.ndarray]:
    odir = TEST_SAMPLES_DIR / video_id / "object_images"
    imgs = []
    for k in [1,2,3]:
        p = odir / f"img_{k}.jpg"
        if p.exists():
            imgs.append(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB))
    return imgs

def get_test_video_path(video_id: str) -> Path:
    return TEST_SAMPLES_DIR / video_id / "drone_video.mp4"

In [77]:
# Cell: (Tùy chọn) Tải checkpoint YOLO đã train
# Bỏ qua cell train YOLO ở trên và chạy cell này nếu bạn đã có checkpoint
# Tìm checkpoint YOLO mới nhất
yolo_exp_dir = WORKDIR / "yolo_train"
try:
    # Tìm thư mục experiment mới nhất (ví dụ: exp_one_shot, exp_one_shot2, ...)
    latest_exp = sorted([p for p in yolo_exp_dir.glob("exp_one_shot*") if p.is_dir()])[-1]
    potential_ckpt = latest_exp / "weights" / "best.pt"

    if 'best_yolo_ckpt' not in locals() and potential_ckpt.exists():
        best_yolo_ckpt = potential_ckpt
        print(f"Đã tìm thấy và tải checkpoint YOLO: {best_yolo_ckpt}")
    elif 'best_yolo_ckpt' in locals():
        print(f"Biến best_yolo_ckpt đã tồn tại: {best_yolo_ckpt}")
    else:
        print(f"Không tìm thấy checkpoint YOLO tại {potential_ckpt}. Vui lòng chạy lại cell huấn luyện.")
except (IndexError, FileNotFoundError):
    print(f"Không tìm thấy thư mục huấn luyện YOLO trong {yolo_exp_dir}. Vui lòng chạy cell huấn luyện.")


# Cell: Tải mô hình tốt nhất cho inference
yolo_inf = YOLO(str(best_yolo_ckpt))

Biến best_yolo_ckpt đã tồn tại: d:\zalo.ai\AeroEyes\workdir_one_shot\yolo_train\exp_one_shot\weights\best.pt


In [78]:
# Cell: Tải mô hình tốt nhất cho inference
yolo_inf = YOLO(str(best_yolo_ckpt))
embed_model.load_state_dict(torch.load(best_embed_path, map_location=DEVICE))
embed_model.eval();

In [79]:
# Cell: Học embedding tham chiếu từ 3 ảnh test (+augment)
def build_ref_embedding_for_video(video_id: str, repeats=8) -> torch.Tensor:
    imgs = load_ref_images_test(video_id)
    assert len(imgs)>0, f"Thiếu ảnh tham chiếu cho {video_id}"
    vecs = []
    with torch.no_grad():
        for im in imgs:
            t = embed_infer_aug(image=im)["image"].to(DEVICE).float()/255.0
            vecs.append(embed_model(t.unsqueeze(0)))
            for _ in range(repeats):
                ta = embed_train_aug(image=im)["image"].to(DEVICE).float()/255.0
                vecs.append(embed_model(ta.unsqueeze(0)))
    z = torch.cat(vecs, dim=0).mean(dim=0)
    z = F.normalize(z, dim=-1)
    return z  # [1, D]

In [80]:
# Cell: Embed các ROI từ boxes
def crop_and_embed_rois(frame_rgb: np.ndarray, boxes_xyxy: np.ndarray) -> torch.Tensor:
    crops = []
    for (x1,y1,x2,y2) in boxes_xyxy.astype(int):
        x1 = max(0, x1); y1 = max(0, y1)
        x2 = max(x1+1, x2); y2 = max(y1+1, y2)
        crop = frame_rgb[y1:y2, x1:x2, :]
        if crop.size == 0:
            crop = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        t = embed_infer_aug(image=crop)["image"]
        crops.append(t)
    if not crops:
        return torch.empty(0, embed_dim, device=DEVICE)
    batch = torch.stack(crops).to(DEVICE).float()/255.0
    with torch.no_grad():
        z = embed_model(batch)
    return z

def combine_scores(yolo_confs: torch.Tensor, sims: torch.Tensor, alpha=0.4) -> torch.Tensor:
    """
    Kết hợp score với logic thông minh:
    - Nếu YOLO conf cao nhưng similarity thấp -> giảm score (có thể là false positive)
    - Nếu cả hai đều hợp lý -> chấp nhận
    - Nếu similarity rất cao -> ưu tiên (có thể là true positive)
    """
    sims01 = (sims + 1.0) / 2.0  # [-1,1] -> [0,1]
    
    # Công thức cơ bản: cân bằng giữa YOLO và similarity
    final = alpha * yolo_confs + (1-alpha) * sims01
    
    # Nếu YOLO conf cao nhưng similarity quá thấp -> phạt (có thể là false positive)
    high_yolo_low_sim = (yolo_confs > 0.3) & (sims < 0.2)
    if high_yolo_low_sim.any():
        final[high_yolo_low_sim] *= 0.5  # Giảm score đi một nửa
    
    # Nếu similarity cao (>= 0.3) -> ưu tiên (có thể là true positive)
    high_sim_mask = sims >= 0.3
    if high_sim_mask.any():
        # Với similarity cao, tăng trọng số cho nó
        final[high_sim_mask] = 0.3 * yolo_confs[high_sim_mask] + 0.7 * sims01[high_sim_mask]
    
    # Nếu cả YOLO và similarity đều cao -> tăng cường
    high_both_mask = (yolo_confs > 0.2) & (sims > 0.3)
    if high_both_mask.any():
        final[high_both_mask] = 0.25 * yolo_confs[high_both_mask] + 0.75 * sims01[high_both_mask]
    
    return final

In [81]:
# Cell: Suy luận 1 video test
def infer_video(video_id: str, 
                alpha=0.4,              # Cân bằng hơn
                conf_thres=0.05,         # GIẢM từ 0.15 -> 0.05 (không bỏ sót detection yếu)
                min_similarity=0.25,    # GIẢM từ 0.5 -> 0.25 (không quá nghiêm ngặt)
                iou_nms=0.5,
                temporal_smooth=True,
                min_final_score=0.35):   # GIẢM từ 0.4 -> 0.35
    """
    Inference với logic thông minh hơn:
    - Không quá nghiêm ngặt với similarity để không bỏ sót true positive
    - Nhưng vẫn lọc false positive bằng cách phạt khi YOLO cao nhưng similarity thấp
    """
    vpath = get_test_video_path(video_id)
    assert vpath.exists(), f"Thiếu video test: {vpath}"
    ref_z = build_ref_embedding_for_video(video_id)  # [1, D]

    cap = cv2.VideoCapture(str(vpath))
    out = []
    
    prev_detection = None
    temporal_decay = 0.85
    
    idx = 0
    while True:
        ok, frame_bgr = cap.read()
        if not ok:
            break
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        H, W = frame_rgb.shape[:2]

        # YOLO inference với threshold thấp hơn để không bỏ sót
        res = yolo_inf.predict(source=frame_rgb, imgsz=640, conf=conf_thres, verbose=False)[0]
        
        current_detections = []
        
        if res.boxes is not None and len(res.boxes) > 0:
            bxyxy = res.boxes.xyxy.cpu().numpy()
            confs = res.boxes.conf.detach().cpu()

            # Embed ROIs
            z_rois = crop_and_embed_rois(frame_rgb, bxyxy)
            if z_rois.size(0) > 0:
                with torch.no_grad():
                    # Tính similarity
                    sims = F.cosine_similarity(z_rois, ref_z.expand_as(z_rois), dim=-1)  # [-1,1]
                    
                    # --- PHẦN SỬA LỖI ---
                    # Chuyển confs sang cùng thiết bị với sims (GPU) để tính toán
                    confs_gpu = confs.to(DEVICE)

                    # BỘ LỌC LINH HOẠT HƠN (tính toán hoàn toàn trên tensor)
                    valid_mask = (
                        (sims >= min_similarity) |  # Similarity đủ cao
                        ((confs_gpu > 0.2) & (sims > 0.15))  # Hoặc cả hai đều hợp lý
                    ) & (
                        ~((confs_gpu > 0.3) & (sims < 0.1))  # Loại bỏ: YOLO cao nhưng similarity quá thấp
                    )
                    
                    if valid_mask.any():
                        # Chuyển mask về CPU để lọc mảng numpy `bxyxy` và tensor cpu `confs`
                        valid_mask_cpu = valid_mask.cpu().numpy()
                        
                        # Chỉ xử lý các detection hợp lệ
                        bxyxy_valid = bxyxy[valid_mask_cpu]
                        confs_valid = confs[valid_mask_cpu]
                        sims_valid = sims[valid_mask]
                        
                        # Kết hợp score với logic thông minh
                        confs_valid_t = confs_valid.to(DEVICE)
                        final_scores = combine_scores(confs_valid_t, sims_valid, alpha=alpha)
                        
                        # Lọc theo min_final_score
                        valid_final_mask = final_scores >= min_final_score
                        if valid_final_mask.any():
                            valid_final_mask_cpu = valid_final_mask.cpu()
                            bxyxy_final = bxyxy_valid[valid_final_mask_cpu.numpy()]
                            final_scores_final = final_scores[valid_final_mask]
                            confs_final = confs_valid[valid_final_mask_cpu.numpy()]
                            sims_final = sims_valid[valid_final_mask]
                            
                            # NMS
                            boxes_t = torch.tensor(bxyxy_final, dtype=torch.float32, device=DEVICE)
                            keep = nms(boxes_t, final_scores_final, iou_nms)
                            kept = keep.detach().cpu().numpy().tolist()

                            for k in kept:
                                x1,y1,x2,y2 = bxyxy_final[k]
                                w = x2 - x1
                                h = y2 - y1
                                x1 = max(0, min(x1, W-1)); y1 = max(0, min(y1, H-1))
                                w = max(1, min(w, W - x1)); h = max(1, min(h, H - y1))
                                
                                det = {
                                    "bbox": [float(x1), float(y1), float(w), float(h)],
                                    "score": float(final_scores_final[k].item()),
                                    "similarity": float(sims_final[k].item()),
                                    "yolo_conf": float(confs_final[k]),
                                    "frame_idx": idx
                                }
                                current_detections.append(det)
                                out.append({
                                    "frame_idx": int(idx),
                                    "bbox": [float(x1), float(y1), float(w), float(h)],
                                    "score": float(final_scores_final[k].item())
                                })
        
        # Temporal smoothing - giữ nguyên logic
        if temporal_smooth and len(current_detections) == 0 and prev_detection is not None:
            prev_bbox = prev_detection["bbox"]
            prev_x, prev_y, prev_w, prev_h = prev_bbox
            
            search_margin = 30
            search_x1 = max(0, prev_x - search_margin)
            search_y1 = max(0, prev_y - search_margin)
            search_x2 = min(W, prev_x + prev_w + search_margin)
            search_y2 = min(H, prev_y + prev_h + search_margin)
            
            search_roi = frame_rgb[int(search_y1):int(search_y2), int(search_x1):int(search_x2), :]
            if search_roi.size > 0:
                search_h, search_w = search_roi.shape[:2]
                if search_h > 0 and search_w > 0:
                    t_search = embed_infer_aug(image=search_roi)["image"]
                    z_search = embed_model(t_search.unsqueeze(0).to(DEVICE).float()/255.0)
                    with torch.no_grad():
                        sim_search = F.cosine_similarity(z_search, ref_z, dim=-1).item()
                    
                    # Threshold thấp hơn cho temporal smoothing
                    if sim_search >= min_similarity * 0.8:  # 0.25 * 0.8 = 0.2
                        decayed_score = prev_detection["score"] * temporal_decay
                        if decayed_score >= min_final_score * 0.7:
                            out.append({
                                "frame_idx": int(idx),
                                "bbox": prev_bbox.copy(),
                                "score": float(decayed_score)
                            })
                            current_detections.append({
                                "bbox": prev_bbox.copy(),
                                "score": float(decayed_score),
                                "similarity": sim_search,
                                "frame_idx": idx
                            })
        
        if len(current_detections) > 0:
            best_det = max(current_detections, key=lambda x: x["score"])
            prev_detection = best_det
        elif prev_detection is not None:
            prev_detection["score"] *= temporal_decay
        
        idx += 1

    cap.release()
    return out

In [83]:
# Cell: Hàm chuyển đổi format submission
def convert_to_submission_format(detections):
    """
    Chuyển đổi từ format hiện tại sang format yêu cầu:
    - frame_idx -> frame
    - bbox: [x, y, w, h] -> x1, y1, x2, y2
    - Loại bỏ score
    - Nhóm detections thành tracks
    """
    if len(detections) == 0:
        return []
    
    # Chuyển đổi format cơ bản
    converted = []
    for det in detections:
        frame = det["frame_idx"]
        x, y, w, h = det["bbox"]
        x1, y1 = int(x), int(y)
        x2, y2 = int(x + w), int(y + h)
        
        converted.append({
            "frame": frame,
            "x1": x1,
            "y1": y1,
            "x2": x2,
            "y2": y2
        })
    
    # Nhóm detections thành tracks dựa trên IoU và khoảng cách frame
    # Sử dụng thuật toán đơn giản: nhóm các detection liên tiếp có IoU > 0.3
    def calculate_iou(bbox1, bbox2):
        """Tính IoU giữa 2 bbox"""
        x1_1, y1_1, x2_1, y2_1 = bbox1["x1"], bbox1["y1"], bbox1["x2"], bbox1["y2"]
        x1_2, y1_2, x2_2, y2_2 = bbox2["x1"], bbox2["y1"], bbox2["x2"], bbox2["y2"]
        
        # Tính diện tích giao nhau
        xi1 = max(x1_1, x1_2)
        yi1 = max(y1_1, y1_2)
        xi2 = min(x2_1, x2_2)
        yi2 = min(y2_1, y2_2)
        
        if xi2 <= xi1 or yi2 <= yi1:
            return 0.0
        
        inter_area = (xi2 - xi1) * (yi2 - yi1)
        box1_area = (x2_1 - x1_1) * (y2_1 - y1_1)
        box2_area = (x2_2 - x1_2) * (y2_2 - y1_2)
        union_area = box1_area + box2_area - inter_area
        
        if union_area == 0:
            return 0.0
        
        return inter_area / union_area
    
    # Sắp xếp theo frame
    converted.sort(key=lambda x: x["frame"])
    
    # Nhóm thành tracks
    tracks = []
    if len(converted) == 0:
        return []
    
    current_track = [converted[0]]
    
    for i in range(1, len(converted)):
        prev_bbox = converted[i-1]
        curr_bbox = converted[i]
        
        # Kiểm tra xem có thuộc cùng track không
        frame_diff = curr_bbox["frame"] - prev_bbox["frame"]
        iou = calculate_iou(prev_bbox, curr_bbox)
        
        # Nếu frame liên tiếp hoặc gần nhau (<= 5 frames) và IoU > 0.3, thì cùng track
        if frame_diff <= 5 and iou > 0.3:
            current_track.append(curr_bbox)
        else:
            # Kết thúc track hiện tại và bắt đầu track mới
            if len(current_track) > 0:
                tracks.append({"bboxes": current_track})
            current_track = [curr_bbox]
    
    # Thêm track cuối cùng
    if len(current_track) > 0:
        tracks.append({"bboxes": current_track})
    
    return tracks

In [84]:
# Cell: Chạy trên toàn bộ public_test và tạo submission.json
test_video_ids = [p.name for p in sorted(TEST_SAMPLES_DIR.glob("*")) if (p / "drone_video.mp4").exists()]
print("Số video test:", len(test_video_ids))

submission = []
for vid in tqdm(test_video_ids, desc="Inference test videos"):
    preds = infer_video(
        vid,
        alpha=0.4,              # Cân bằng giữa YOLO và similarity
        conf_thres=0.05,       # GIẢM để không bỏ sót detection yếu
        min_similarity=0.25,   # GIẢM để không quá nghiêm ngặt
        iou_nms=0.5,
        temporal_smooth=True,
        min_final_score=0.35   # GIẢM một chút
    )
    
    # Chuyển đổi sang format yêu cầu
    formatted_detections = convert_to_submission_format(preds)
    
    submission.append({
        "video_id": vid,
        "detections": formatted_detections  # Đã chuyển đổi format
    })

SUBMIT_PATH = WORKDIR / "submission.json"
with open(SUBMIT_PATH, "w") as f:
    json.dump(submission, f, ensure_ascii=False, indent=2)

print("Saved submission to:", SUBMIT_PATH)

Số video test: 6


Inference test videos: 100%|██████████| 6/6 [08:44<00:00, 87.46s/it]

Saved submission to: d:\zalo.ai\AeroEyes\workdir_one_shot\submission.json


In [86]:
# Cell: Visualize kết quả trên tập test
import cv2
import json
from pathlib import Path
from tqdm import tqdm
import random

# --- Cấu hình ---
SUBMISSION_PATH = WORKDIR / "submission.json"
VIS_OUTPUT_DIR = WORKDIR / "test_visualizations"
VIS_OUTPUT_DIR.mkdir(exist_ok=True)
NUM_VIDEOS_TO_VISUALIZE = 2 # Số lượng video muốn visualize, đặt thành -1 để visualize tất cả

# --- Load submission ---
assert SUBMISSION_PATH.exists(), f"Không tìm thấy file submission: {SUBMISSION_PATH}"
with open(SUBMISSION_PATH, "r") as f:
    submission_data = json.load(f)

# --- Gom nhóm detection theo frame_idx để truy cập nhanh hơn ---
def group_dets_by_frame(tracks):
    grouped = {}
    # Lặp qua từng track
    for track in tracks:
        # Lặp qua từng bbox trong track
        for det in track["bboxes"]:
            fidx = det["frame"] # Sửa: key là 'frame'
            # Chuyển đổi bbox từ x1,y1,x2,y2 sang x,y,w,h để hàm draw_boxes dùng được
            x1, y1, x2, y2 = det["x1"], det["y1"], det["x2"], det["y2"]
            w = x2 - x1
            h = y2 - y1
            
            # Tạo một dict mới với format mà draw_boxes mong đợi
            new_det_format = {
                "bbox": [x1, y1, w, h],
                "score": 1.0 # Score không có trong submission cuối, đặt tạm là 1.0
            }
            grouped.setdefault(fidx, []).append(new_det_format)
    return grouped

# --- Vẽ bounding box lên frame ---
def draw_boxes(frame, detections):
    for det in detections:
        box = det['bbox']
        score = det['score']
        x, y, w, h = map(int, box)
        
        # Vẽ hộp
        cv2.rectangle(frame, (x, y), (x + w, y + h), (36, 255, 12), 2)
        
        # Chuẩn bị text (score)
        label = f"{score:.2f}"
        (label_width, label_height), baseline = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
        
        # Vẽ nền cho text
        cv2.rectangle(frame, (x, y - label_height - baseline), (x + label_width, y), (36, 255, 12), -1)
        
        # Vẽ text
        cv2.putText(frame, label, (x, y - baseline), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 2)
    return frame

# --- Xử lý từng video ---
videos_to_process = submission_data
if NUM_VIDEOS_TO_VISUALIZE != -1:
    videos_to_process = submission_data[:NUM_VIDEOS_TO_VISUALIZE]

for item in tqdm(videos_to_process, desc="Visualizing test videos"):
    video_id = item["video_id"]
    detections = item["detections"]
    
    # Nhóm detections theo frame
    frame_to_dets = group_dets_by_frame(detections)
    
    # Mở video gốc
    video_path = TEST_SAMPLES_DIR / video_id / "drone_video.mp4"
    cap = cv2.VideoCapture(str(video_path))
    
    # Chuẩn bị video output
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    output_path = VIS_OUTPUT_DIR / f"{video_id}_prediction.mp4"
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(str(output_path), fourcc, fps, (w, h))
    
    frame_idx = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        # Lấy detection cho frame hiện tại và vẽ
        if frame_idx in frame_to_dets:
            frame = draw_boxes(frame, frame_to_dets[frame_idx])
            
        writer.write(frame)
        frame_idx += 1
        
    cap.release()
    writer.release()
    print(f"Đã lưu video visualize cho '{video_id}' tại: {output_path}")

print("\nHoàn tất visualization!")

Visualizing test videos:   0%|          | 0/2 [00:00<?, ?it/s]

Visualizing test videos:  50%|█████     | 1/2 [00:24<00:24, 24.09s/it]

Đã lưu video visualize cho 'BlackBox_0' tại: d:\zalo.ai\AeroEyes\workdir_one_shot\test_visualizations\BlackBox_0_prediction.mp4


Visualizing test videos: 100%|██████████| 2/2 [00:52<00:00, 26.28s/it]

Đã lưu video visualize cho 'BlackBox_1' tại: d:\zalo.ai\AeroEyes\workdir_one_shot\test_visualizations\BlackBox_1_prediction.mp4

Hoàn tất visualization!


In [82]:
# Cell: Chạy và visualize chỉ video đầu tiên của tập test
# Lấy video đầu tiên
test_video_ids = [p.name for p in sorted(TEST_SAMPLES_DIR.glob("*")) if (p / "drone_video.mp4").exists()]

if not test_video_ids:
    print("Không tìm thấy video nào trong thư mục test.")
else:
    target_video_id = test_video_ids[0]
    print(f"Bắt đầu inference cho video: {target_video_id}")

    # 1. Chạy inference trên 1 video
    preds = infer_video(
        target_video_id,
        alpha=0.4,
        conf_thres=0.05,
        min_similarity=0.25,
        iou_nms=0.5,
        temporal_smooth=True,
        min_final_score=0.35
    )
    
    print(f"Inference hoàn tất. Tìm thấy {len(preds)} detections.")

    # 2. Visualize kết quả của video đó
    VIS_OUTPUT_DIR = WORKDIR / "test_visualizations"
    VIS_OUTPUT_DIR.mkdir(exist_ok=True)

    # Gom nhóm detection theo frame_idx để truy cập nhanh hơn
    def group_dets_by_frame(detections):
        grouped = {}
        for det in detections:
            fidx = det["frame_idx"]
            grouped.setdefault(fidx, []).append(det)
        return grouped

    # Vẽ bounding box lên frame
    def draw_boxes(frame, detections):
        for det in detections:
            box = det['bbox']
            score = det['score']
            x, y, w, h = map(int, box)
            cv2.rectangle(frame, (x, y), (x + w, y + h), (36, 255, 12), 2)
            label = f"{score:.2f}"
            (label_width, label_height), baseline = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
            cv2.rectangle(frame, (x, y - label_height - baseline), (x + label_width, y), (36, 255, 12), -1)
            cv2.putText(frame, label, (x, y - baseline), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 2)
        return frame

    # Nhóm detections theo frame
    frame_to_dets = group_dets_by_frame(preds)
    
    # Mở video gốc
    video_path = TEST_SAMPLES_DIR / target_video_id / "drone_video.mp4"
    cap = cv2.VideoCapture(str(video_path))
    
    # Chuẩn bị video output
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    output_path = VIS_OUTPUT_DIR / f"{target_video_id}_single_prediction.mp4"
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(str(output_path), fourcc, fps, (w, h))
    
    print(f"Bắt đầu ghi video visualize tới: {output_path}")
    
    frame_idx = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        if frame_idx in frame_to_dets:
            frame = draw_boxes(frame, frame_to_dets[frame_idx])
            
        writer.write(frame)
        frame_idx += 1
        
    cap.release()
    writer.release()
    print(f"Đã lưu video visualize thành công!")

Bắt đầu inference cho video: BlackBox_0
Inference hoàn tất. Tìm thấy 402 detections.
Bắt đầu ghi video visualize tới: d:\zalo.ai\AeroEyes\workdir_one_shot\test_visualizations\BlackBox_0_single_prediction.mp4
Đã lưu video visualize thành công!
